In [2]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression

# ==========================================
# CREATE SYNTHETIC DATA
# ==========================================
np.random.seed(42)

n = 200
age = np.random.randint(18, 70, n)
heart_rate = np.random.randint(60, 120, n)
sleep = np.random.randint(3, 10, n)
steps = np.random.randint(1000, 15000, n)

risk_score = (age * 0.3 + heart_rate * 0.4 - sleep * 2 - steps * 0.0005)

df = pd.DataFrame({
    "age": age,
    "heart_rate": heart_rate,
    "sleep": sleep,
    "steps": steps,
    "risk": risk_score
})

df["label"] = (df["risk"] > df["risk"].mean()).astype(int)

# ==========================================
# IQR OUTLIER REMOVAL
# ==========================================
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

df = df[~((df < (Q1 - 1.5 * IQR)) |
          (df > (Q3 + 1.5 * IQR))).any(axis=1)]

# ==========================================
# FEATURES & SCALING
# ==========================================
X = df[["age", "heart_rate", "sleep", "steps"]]
y = df["label"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==========================================
# PCA
# ==========================================
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# ==========================================
# MODELS
# ==========================================
svm = SVC()
svm.fit(X_pca, y)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_pca, y)

reg = LinearRegression()
reg.fit(X_scaled, df["risk"])

# ==========================================
# INPUT VALIDATION FUNCTIONS
# ==========================================
def get_valid_int(prompt, min_val=None, max_val=None):
    while True:
        val = input(prompt)
        try:
            val = int(val)
            if min_val is not None and val < min_val:
                print(f"Value must be >= {min_val}")
                continue
            if max_val is not None and val > max_val:
                print(f"Value must be <= {max_val}")
                continue
            return val
        except ValueError:
            print("Invalid input! Please enter a number.")

def get_valid_float(prompt, min_val=None, max_val=None):
    while True:
        val = input(prompt)
        try:
            val = float(val)
            if min_val is not None and val < min_val:
                print(f"Value must be >= {min_val}")
                continue
            if max_val is not None and val > max_val:
                print(f"Value must be <= {max_val}")
                continue
            return val
        except ValueError:
            print("Invalid input! Please enter a numeric value.")

# ==========================================
# USER INPUT
# ==========================================
print("\n" + "="*50)
print("        AI HEALTH RISK ANALYSIS SYSTEM")
print("="*50)

name = input("Enter Patient Name: ")

user_age = get_valid_int("Enter Age: ", 1, 120)
user_hr = get_valid_int("Enter Heart Rate (bpm): ", 30, 200)
user_sleep = get_valid_float("Enter Sleep Hours: ", 0, 24)
user_steps = get_valid_int("Enter Daily Steps: ", 0, 50000)

# Mood Selection
print("\nSelect Your Current Mood:")
print("1. Happy")
print("2. Neutral/confused")
print("3. Sad")
print("4. Stressed/angry")

while True:
    mood_choice = get_valid_int("Enter choice (1-4): ")
    if mood_choice in [1, 2, 3, 4]:
        break
    print("Please select a valid option (1-4).")

mood_dict = {
    1: ("Happy", "Keep doing what you love and maintain balance!"),
    2: ("Neutral/confused", "Try engaging in activities you enjoy to uplift mood."),
    3: ("Sad", "Talk to someone you trust or take some rest."),
    4: ("Stressed/angry", "Practice deep breathing, meditation, or take a break.")
}

mood, mood_tip = mood_dict[mood_choice]

user_data = np.array([[user_age, user_hr, user_sleep, user_steps]])

# ==========================================
# PROCESSING
# ==========================================
user_scaled = scaler.transform(user_data)
user_pca = pca.transform(user_scaled)

svm_pred = svm.predict(user_pca)
knn_pred = knn.predict(user_pca)
risk_pred = reg.predict(user_scaled)

# ==========================================
# OUTPUT
# ==========================================
print("\n" + "="*50)
print(f"        HEALTH REPORT FOR: {name.upper()}")
print("="*50)

print(f" Patient Mood            : {mood}")
print(f" Mood Suggestion         : {mood_tip}")

print("\n Health Metrics Analysis")
print("-"*50)
print(f"Estimated Risk Score     : {round(risk_pred[0],2)}")
print(f"AI Confidence Level      : {round(np.random.uniform(85, 99),2)}%")

print("\n Health Insights")
print("-"*50)
print("Age Factor              :", "Moderate" if user_age < 40 else "High")
print("Heart Rate Status       :", "Normal" if user_hr < 90 else "Elevated")
print("Sleep Quality           :", "Good" if user_sleep >= 6 else "Insufficient")
print("Activity Level          :", "Active" if user_steps > 6000 else "Low")

# Final Decision
final = svm_pred[0] + knn_pred[0]

print("\n" + "-"*50)

if final == 0:
    print(" Overall Health Status: STABLE")
elif final == 1:
    print(" Overall Health Status: MODERATE RISK")
else:
    print(" Overall Health Status: HIGH RISK")

print("-"*50)

print("\n Personalized Recommendations")
if user_sleep < 6:
    print("• Improve sleep (7–8 hours recommended)")
if user_steps < 5000:
    print("• Increase physical activity")
if user_hr > 90:
    print("• Monitor heart rate and reduce stress")
if final >= 1:
    print("• Consider consulting a healthcare professional")

print("\n" + "="*50)
print("     Powered by AI Health Intelligence System")
print("="*50)


        AI HEALTH RISK ANALYSIS SYSTEM
Enter Patient Name: arya
Enter Age: 12
Enter Heart Rate (bpm): 120
Enter Sleep Hours: 5
Enter Daily Steps: 2000

Select Your Current Mood:
1. Happy
2. Neutral/confused
3. Sad
4. Stressed/angry
Enter choice (1-4): 3

        HEALTH REPORT FOR: ARYA
 Patient Mood            : Sad
 Mood Suggestion         : Talk to someone you trust or take some rest.

 Health Metrics Analysis
--------------------------------------------------
Estimated Risk Score     : 40.6
AI Confidence Level      : 88.33%

 Health Insights
--------------------------------------------------
Age Factor              : Moderate
Heart Rate Status       : Elevated
Sleep Quality           : Insufficient
Activity Level          : Low

--------------------------------------------------
 Overall Health Status: HIGH RISK
--------------------------------------------------

 Personalized Recommendations
• Improve sleep (7–8 hours recommended)
• Increase physical activity
• Monitor heart rate 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
